# SDTMIG_Statements — what SDTMIG v3.4 says about test facets and domain routing

Builds `sdtm-domain-reference/machine_actionable/SDTMIG_Statements.xlsx`: selected statements from
SDTMIG v3.4, quoted verbatim with section, item and page, and the variable-table rows for every
variable they name.

Facts with citations only. Nothing is classified, interpreted or decided here. The only hand-kept
input is the selection below: which statements are quoted, and whether each was selected as a
**facet** statement (what describes a test beside --TEST) or a **routing** statement (which domain
a kind of test goes to). `Selected_As` records why a row was selected, not what the text means.

## Inputs (gitignored; downloaded with a cdisc.org sign-in)

| File | Used for |
|---|---|
| `sdtm-domain-reference/downloads/SDTMIG v3.4-FINAL_2022-07-21.pdf` | Domain Assumptions and §4.1.9, quoted from the text pdfminer.six extracts |
| `sdtm-domain-reference/downloads/SDTMIG_v3.4.csv` | CDISC Notes, and the `Variables` sheet (rows as published) |
| `sdtm-domain-reference/downloads/SDTM_v2.0.csv` | `Model_Variables_Named`: the generic `--` variables of each observation class |
| `Reference_Versions.json` | SDTMIG pin |

Needs `pdfminer.six` (`pip install pdfminer.six`). Reading the PDF takes about a minute.

The run **fails** when: the SDTMIG pin is not 3.4 or the SDTM pin not 2.0; a CSV or the PDF is another version; a printed page
number differs from the PDF page; a selected statement is not found, or found more than once.

In [1]:
import hashlib
import json
import re
from bisect import bisect_right
from datetime import datetime
from pathlib import Path

import pandas as pd
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from pdfminer.high_level import extract_text

BASE_DIR = Path.cwd().parent          # sdtm-domain-reference/
REPO_ROOT = BASE_DIR.parent           # cdisc-for-ai/
VERSIONS = json.loads((REPO_ROOT / 'Reference_Versions.json').read_text(encoding='utf-8'))
IG_PDF = BASE_DIR / 'downloads' / 'SDTMIG v3.4-FINAL_2022-07-21.pdf'
IG_CSV = BASE_DIR / 'downloads' / 'SDTMIG_v3.4.csv'
MODEL_CSV = BASE_DIR / 'downloads' / 'SDTM_v2.0.csv'
OUTPUT_FILE = BASE_DIR / 'machine_actionable' / 'SDTMIG_Statements.xlsx'
SOURCE = 'SDTMIG v3.4 (2021-11-29)'

for path in [IG_PDF, IG_CSV, MODEL_CSV]:
    if not path.exists():
        raise FileNotFoundError(path)
if VERSIONS['SDTMIG'] != '3.4':
    raise RuntimeError(f'This selection is for SDTMIG v3.4; the pin is {VERSIONS["SDTMIG"]}: re-check every locator')
if VERSIONS['SDTM'] != '2.0':
    raise RuntimeError(f'The SDTM pin is {VERSIONS["SDTM"]}; this notebook reads SDTM v2.0')

ig = pd.read_csv(IG_CSV, dtype=str, keep_default_na=False)
if set(ig['Version']) != {'SDTMIG v3.4'}:
    raise RuntimeError(f'{IG_CSV.name}: Version {sorted(set(ig["Version"]))}')
model = pd.read_csv(MODEL_CSV, dtype=str, keep_default_na=False)
if set(model['Version']) != {'SDTM v2.0'}:
    raise RuntimeError(f'{MODEL_CSV.name}: Version {sorted(set(model["Version"]))}')
print(f'{IG_CSV.name}: {len(ig):,} variables in {ig["Dataset Name"].nunique()} datasets')
print(f'{MODEL_CSV.name}: {len(model):,} variables')

SDTMIG_v3.4.csv: 1,917 variables in 63 datasets
SDTM_v2.0.csv: 527 variables


## 1. The selection (hand-kept)

Three kinds of locator:

- `assumption` — a domain Assumptions heading as printed, the item number, and the lettered
  sub-item (`''` = the item's own text before its first sub-item). A sub-item keeps its roman
  sub-items (MB 1c holds i–iv).
- `anchor` — a passage outside the Assumptions (§4.1.9), from a start string to an end string,
  both inclusive; each must occur once in the section.
- `notes` — the CDISC Notes of a variable in the variable table (CSV), as published.

In [2]:
SELECTION = [
    # Statement_ID, Selected_As, locator
    ('4.1.9-1', 'facet', ('anchor', '4.1.9 Assigning Natural Keys in the Metadata',
                          'Section 3.2, Using the CDISC Domain Models', 'because it forms part of the natural key.')),
    ('4.1.9-2', 'facet', ('anchor', '4.1.9 Assigning Natural Keys in the Metadata',
                          'This approach becomes very useful in a Findings domain', 'to fully identify the result.')),
    ('4.1.9-3', 'facet', ('anchor', '4.1.9 Assigning Natural Keys in the Metadata',
                          'When CDISC Controlled Terminology for a test is not available', 'to fully express the test detail.')),
    ('4.1.9-4', 'facet', ('anchor', '4.1.9 Assigning Natural Keys in the Metadata',
                          'The MKTESTCD value "SGBESCR" is a generic test code', 'to describe the uniqueness of the test.')),
    ('LB-A8', 'facet', ('assumption', 'LB – Assumptions', 8, '')),
    ('LBRESSCL-Notes', 'facet', ('notes', 'LB', 'LBRESSCL')),
    ('ISBDAGNT-Notes', 'facet', ('notes', 'IS', 'ISBDAGNT')),
    ('IS-A7', 'facet', ('assumption', 'IS – Assumptions', 7, '')),
    ('IS-A7a', 'facet', ('assumption', 'IS – Assumptions', 7, 'a')),
    ('IS-A7b', 'facet', ('assumption', 'IS – Assumptions', 7, 'b')),
    ('IS-A8', 'facet', ('assumption', 'IS – Assumptions', 8, '')),
    ('GF-A5', 'facet', ('assumption', 'GF – Assumptions', 5, '')),
    ('GF-A5a', 'facet', ('assumption', 'GF – Assumptions', 5, 'a')),
    ('GF-A5b', 'facet', ('assumption', 'GF – Assumptions', 5, 'b')),
    ('MB-A1c', 'facet', ('assumption', 'MB – Assumptions', 1, 'c')),
    ('MBTSTDTL-Notes', 'facet', ('notes', 'MB', 'MBTSTDTL')),
    ('IS-A2', 'routing', ('assumption', 'IS – Assumptions', 2, '')),
    ('IS-A5', 'routing', ('assumption', 'IS – Assumptions', 5, '')),
    ('IS-A6', 'routing', ('assumption', 'IS – Assumptions', 6, '')),
    ('GF-A1', 'routing', ('assumption', 'GF – Assumptions', 1, '')),
    ('GF-A2', 'routing', ('assumption', 'GF – Assumptions', 2, '')),
    ('GF-A2a', 'routing', ('assumption', 'GF – Assumptions', 2, 'a')),
    ('GF-A2b', 'routing', ('assumption', 'GF – Assumptions', 2, 'b')),
    ('GF-A2c', 'routing', ('assumption', 'GF – Assumptions', 2, 'c')),
    ('GF-A2d', 'routing', ('assumption', 'GF – Assumptions', 2, 'd')),
    ('GF-A8', 'routing', ('assumption', 'GF – Assumptions', 8, '')),
    ('MB-A1', 'routing', ('assumption', 'MB – Assumptions', 1, '')),
    ('MB-A1a', 'routing', ('assumption', 'MB – Assumptions', 1, 'a')),
    ('MB-A1b', 'routing', ('assumption', 'MB – Assumptions', 1, 'b')),
    ('PCTESTCD-Notes', 'routing', ('notes', 'PC', 'PCTESTCD')),
    ('PCTEST-Notes', 'routing', ('notes', 'PC', 'PCTEST')),
    ('RS-DR-A1', 'routing', ('assumption', 'RS – Disease Response Use Case Assumptions', 1, '')),
]
ids = [s[0] for s in SELECTION]
if len(ids) != len(set(ids)):
    raise ValueError('Duplicate Statement_ID in SELECTION')
print(f'{len(SELECTION)} statements selected')

32 statements selected


## 2. Read the PDF

One list of (page, line) with the page furniture (running header, copyright footer, date, page
number) removed. Every printed page number must equal the PDF page, so `Page` is both.
Lines are joined with one space; a line ending in `-` is joined without one (`NON-STUDY-` +
`RELATED`). Runs of whitespace become one space. Nothing else is changed: quotes, bullets and
spacing around punctuation stay as extracted.

In [3]:
pdf_text = extract_text(IG_PDF)
pages = pdf_text.split('\f')
if '3.4 Final' not in pages[0] and '3.4 Final' not in pages[1]:
    raise RuntimeError(f'{IG_PDF.name} does not read as SDTMIG 3.4 Final')

FURNITURE = re.compile(r'^(CDISC Study Data Tabulation Model Implementation Guide: Human Clinical Trials \(3\.4 Final\)'
                       r'|© \d{4} Clinical Data Interchange Standards Consortium, Inc\. All rights reserved'
                       r'|\d{4}-\d{2}-\d{2}|Page \d+)$')
pdf_lines = []
for p, page in enumerate(pages, start=1):
    for raw in page.split('\n'):
        l = raw.strip()
        m = re.fullmatch(r'Page (\d+)', l)
        if m and int(m.group(1)) != p:
            raise RuntimeError(f'Printed page {m.group(1)} on PDF page {p}')
        if l and not FURNITURE.match(l):
            pdf_lines.append((p, l))
print(f'{IG_PDF.name}: {len(pages)} pages, {len(pdf_lines):,} lines')


def join_lines(ls):
    out = ''
    for l in ls:
        out = out + l if out.endswith('-') or not out else out + ' ' + l
    return re.sub(r'\s+', ' ', out).strip()

SDTMIG v3.4-FINAL_2022-07-21.pdf: 462 pages, 65,808 lines


## 3. Locate the statements

`Section` = the numbered heading the statement sits under, plus the Assumptions heading.
An item or sub-item marker counts only in sequence (1, 2, 3 …; a, b, c …; i, ii, iii …), so a
number or letter that happens to start a line of prose is not taken as a marker.

In [4]:
NUMBERED = re.compile(r'^(\d+(?:\.\d+)+)\s+([A-Z][^.]*)$')   # body headings; TOC lines carry dot leaders
ASSUMPTION_HEAD = re.compile(r'^[A-Z]{2} – .*Assumptions$')
BLOCK_END = re.compile(r'^[A-Z]{2} – ')
MARKER = re.compile(r'^([0-9]{1,2}|[a-z]|i|ii|iii|iv|v|vi|vii|viii|ix|x)\.(?:\s+(.*))?$')
ROMAN = ['i', 'ii', 'iii', 'iv', 'v', 'vi', 'vii', 'viii', 'ix', 'x']


def numbered_heading_before(idx):
    for j in range(idx - 1, -1, -1):
        m = NUMBERED.match(pdf_lines[j][1])
        if m:
            return f'{m.group(1)} {m.group(2).strip()}'
    raise RuntimeError(f'No numbered heading before line {idx}')


def parse_assumptions(heading):
    """Return {(item, sub): (page, text)} for the one Assumptions block with this heading."""
    starts = [i for i, (_, l) in enumerate(pdf_lines) if l == heading]
    if len(starts) != 1:
        raise RuntimeError(f'Heading "{heading}" found {len(starts)} times')
    start = starts[0]
    section = f'{numbered_heading_before(start)}; {heading}'
    parts, item, letter, roman, key = {}, 0, '', 0, None
    for p, l in pdf_lines[start + 1:]:
        if BLOCK_END.match(l) or NUMBERED.match(l):
            break
        m = MARKER.match(l)
        tok, rest = (m.group(1), m.group(2) or '') if m else (None, None)
        if tok is not None and tok.isdigit() and int(tok) == item + 1:
            item, letter, roman = item + 1, '', 0
            key = (item, '')
        elif tok is not None and item and tok in ROMAN and letter and roman < len(ROMAN) and tok == ROMAN[roman] \
                and not (len(tok) == 1 and letter == chr(ord(tok) - 1)):
            roman += 1
            parts[key][1].append(f'{tok}. ')
            if rest:
                parts[key][1].append(rest)
            continue
        elif tok is not None and item and len(tok) == 1 and tok.isalpha() and tok == chr(ord(letter or '`') + 1):
            letter, roman = tok, 0
            key = (item, letter)
        else:
            if key is not None:
                parts[key][1].append(l)
            continue
        parts[key] = (p, [rest] if rest else [])
    if not parts:
        raise RuntimeError(f'No items under "{heading}"')
    return section, {k: (p, join_lines(v)) for k, (p, v) in parts.items()}


def anchored(heading, start_text, end_text):
    heads = [i for i, (_, l) in enumerate(pdf_lines) if l == heading]
    if len(heads) != 1:
        raise RuntimeError(f'Heading "{heading}" found {len(heads)} times')
    h = heads[0]
    end = next(i for i in range(h + 1, len(pdf_lines)) if NUMBERED.match(pdf_lines[i][1]))
    offsets, text = [], ''
    for p, l in pdf_lines[h + 1:end]:
        offsets.append((len(text), p))
        text = text + l if text.endswith('-') or not text else text + ' ' + l
    starts = [m.start() for m in re.finditer(re.escape(start_text), text)]
    if len(starts) != 1:
        raise RuntimeError(f'"{start_text}" found {len(starts)} times under {heading}')
    s = starts[0]
    e = text.find(end_text, s)
    if e < 0:
        raise RuntimeError(f'"{end_text}" not found after "{start_text}"')
    page = offsets[bisect_right([o for o, _ in offsets], s) - 1][1]
    return heading, page, re.sub(r'\s+', ' ', text[s:e + len(end_text)]).strip()


blocks = {}
rows = []
for sid, selected_as, loc in SELECTION:
    kind = loc[0]
    if kind == 'assumption':
        _, heading, item, sub = loc
        if heading not in blocks:
            blocks[heading] = parse_assumptions(heading)
        section, parts = blocks[heading]
        if (item, sub) not in parts:
            raise RuntimeError(f'{sid}: item {item}{sub} not under "{heading}"')
        page, text = parts[(item, sub)]
        rows.append((sid, selected_as, section, f'{item}{sub}', str(page), heading[:2], text))
    elif kind == 'anchor':
        _, heading, a, b = loc
        section, page, text = anchored(heading, a, b)
        rows.append((sid, selected_as, section, '', str(page), '', text))
    elif kind == 'notes':
        _, dataset, variable = loc
        hit = ig[(ig['Dataset Name'] == dataset) & (ig['Variable Name'] == variable)]
        if len(hit) != 1:
            raise RuntimeError(f'{sid}: {dataset}.{variable} found {len(hit)} times in {IG_CSV.name}')
        rows.append((sid, selected_as, 'Variable table, CDISC Notes', '', '', dataset, hit['CDISC Notes'].iloc[0]))
    else:
        raise ValueError(f'{sid}: unknown locator {kind}')

statements = pd.DataFrame(rows, columns=['Statement_ID', 'Selected_As', 'Section', 'Item', 'Page', 'Domain', 'Text'])
statements.insert(2, 'Source', SOURCE)
empty = statements[statements['Text'] == '']
if len(empty):
    raise RuntimeError(f'Empty text: {list(empty["Statement_ID"])}')
for heading, (section, parts) in blocks.items():
    print(f'{section}: items {sorted({k[0] for k in parts})}')

6.3.5.6 Laboratory Test Results (LB); LB – Assumptions: items [1, 2, 3, 4, 5, 6, 7, 8]
6.3.5.5 Immunogenicity Specimen Assessments (IS); IS – Assumptions: items [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
6.3.5.4 Genomics Findings (GF); GF – Assumptions: items [1, 2, 3, 4, 5, 6, 7, 8, 9]
6.3.5.7.1 Microbiology Specimen (MB); MB – Assumptions: items [1, 2, 3, 4]
6.3.9.3.1 Disease Response Use Case; RS – Disease Response Use Case Assumptions: items [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]


## 4. Variables and domains named

`Variables_Named`: every word in the text that is a variable of the dataset its first two letters
name, in the SDTMIG v3.4 variable table (e.g. LBTEST in LB). Generic `--` forms and variables of
no single dataset (STUDYID, USUBJID, VISITNUM, SPDEVID, NHOID) are not listed. `Domains_Named`:
dataset codes written as `(XX)` or `XX domain`. Both by pattern, in order of first appearance.

`Model_Variables_Named`: words that are not in the dataset's SDTMIG v3.4 table but whose `--` form
is an SDTM v2.0 variable of the dataset's observation class (e.g. LBPOS: `--POS`, Findings).
Other words that start with a dataset code are printed for review, not written.

In [5]:
ig_vars = set(zip(ig['Dataset Name'], ig['Variable Name']))
datasets = set(ig['Dataset Name'])
dataset_class = dict(zip(ig['Dataset Name'], ig['Class']))
model_vars = set(zip(model['Class'], model['Variable Name']))
WORD = re.compile(r'(?<![A-Za-z0-9-])[A-Z][A-Z0-9]{3,7}(?![A-Za-z0-9])')
DOMAIN_REF = re.compile(r'\(([A-Z]{2})\)|\b([A-Z]{2}) domain\b')

not_in_table = {}


def variables_named(r, in_model=False):
    out = []
    for w in WORD.findall(r['Text']):
        if (w[:2], w) in ig_vars:
            if not in_model:
                out.append(w)
        elif w[:2] in datasets and (dataset_class[w[:2]], '--' + w[2:]) in model_vars:
            if in_model:
                out.append(w)
        elif w[:2] in datasets and not in_model:
            not_in_table.setdefault(w, set()).add(r['Statement_ID'])
    return '; '.join(dict.fromkeys(out))


def domains_named(text):
    found = [a or b for a, b in DOMAIN_REF.findall(text)]
    return '; '.join(dict.fromkeys(d for d in found if d in datasets))


statements.insert(7, 'Variables_Named', statements.apply(variables_named, axis=1))
statements.insert(8, 'Model_Variables_Named', statements.apply(variables_named, axis=1, in_model=True))
statements.insert(9, 'Domains_Named', statements['Text'].map(domains_named))
print('Words that start with a dataset code, in neither the IG table nor the model (not written):')
for w, sids in sorted(not_in_table.items()):
    print(f'  {w}: {", ".join(sorted(sids))}')

named_in = {}
for r in statements.itertuples(index=False):
    for v in filter(None, r.Variables_Named.split('; ')):
        named_in.setdefault(v, []).append(r.Statement_ID)
variables = ig[ig['Variable Name'].isin(named_in) & (ig['Dataset Name'] == ig['Variable Name'].str[:2])].copy()
variables['Named_In'] = variables['Variable Name'].map(lambda v: '; '.join(named_in[v]))
variables = (variables.assign(_o=variables['Variable Order'].astype(int))
             .sort_values(['Dataset Name', '_o']).drop(columns='_o').reset_index(drop=True))
if set(variables['Variable Name']) != set(named_in):
    raise RuntimeError(f'Named but not in table: {set(named_in) - set(variables["Variable Name"])}')
print(f'Statements: {len(statements)} ({statements["Selected_As"].value_counts().to_dict()}); variables: {len(variables)}')
print(statements[['Statement_ID', 'Item', 'Page', 'Variables_Named', 'Model_Variables_Named', 'Domains_Named']].to_string(index=False))

Words that start with a dataset code, in neither the IG table nor the model (not written):
  COLONY: MB-A1c
  CONFIRM: IS-A8
  COUNT: MB-A1c
  ISBDAGT: IS-A7, IS-A7a
  MICROORG: IS-A7
  PRESENT: MB-A1a
  SCREEN: IS-A8
  SUBTYPE: MB-A1b
Statements: 32 ({'facet': 16, 'routing': 16}); variables: 34
  Statement_ID Item Page                                                                                                                                          Variables_Named Model_Variables_Named Domains_Named
       4.1.9-1        26                                                                                                                                                                                             
       4.1.9-2        27                                                                                                                                                   MKTEST                                    
       4.1.9-3        27                                     

## 5. Write the workbook

In [6]:
HEADER_FONT = Font(name='Arial', bold=True, size=10, color='FFFFFF')
DATA_FONT = Font(name='Arial', size=10)
GREY = PatternFill('solid', fgColor='7F7F7F')
PALE = PatternFill('solid', fgColor='F2F2F2')
BAND = PatternFill('solid', fgColor='595959')
THIN = Side(style='thin', color='999999')
BORDER = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)
WRAP = Alignment(wrap_text=True, vertical='top')
WIDE = {'Text': 120, 'CDISC Notes': 120, 'Section': 45, 'Codelist Submission Values': 40}

n_by = statements['Selected_As'].value_counts()
readme_lines = [
    'SDTMIG_Statements — what SDTMIG v3.4 says about test facets and domain routing',
    '',
    'PURPOSE',
    'Selected statements from SDTMIG v3.4, quoted verbatim with section, item and page, so a consumer can',
    '  cite the IG in its own words: how a test is described by a generic --TEST plus qualifier variables',
    '  (facet statements) and which domain a kind of test goes to (routing statements). With the variable-table',
    '  rows of every variable the statements name.',
    '  Facts with citations only: nothing is classified, interpreted or decided here.',
    '',
    'SCOPE',
    f'{len(statements)} statements: {n_by.get("facet", 0)} selected as facet, {n_by.get("routing", 0)} as routing. '
    f'{len(variables)} variables.',
    '  The selection is hand-kept in the notebook; the text is not. Selected_As records why a row was selected,',
    '  not what the text means. Other IG statements are not here.',
    '',
    'SOURCE VERSIONS',
    f'SDTMIG v{VERSIONS["SDTMIG"]} (2021-11-29) / SDTM v{VERSIONS["SDTM"]} · pinned in Reference_Versions.json (repo root).',
    '  As keys for programs: sheet Provenance.',
    f'Generated: {datetime.now():%Y-%m-%d %H:%M}',
    'Notebook: sdtm-domain-reference/notebooks/SDTMIG_Statements.ipynb',
    'Inputs: SDTMIG v3.4-FINAL_2022-07-21.pdf, SDTMIG_v3.4.csv and SDTM_v2.0.csv (CDISC, downloaded with a cdisc.org sign-in;',
    '  not in the repo).',
    'Quoted text © 2021 Clinical Data Interchange Standards Consortium, Inc. Exploratory work, not a CDISC product.',
    '',
    'KNOWN GAPS',
    'Text is as pdfminer.six extracts it, joined into one paragraph: line breaks, indentation and list layout',
    '  are lost; quotes, bullets and spacing around punctuation are kept as extracted (e.g. "MKHNDPOS".This).',
    'CDISC Notes are quoted from the CSV as published, including its literal \\n paragraph markers.',
    'Variables_Named and Domains_Named are found by pattern (see below), not read by a person. Generic --',
    '  forms and identifiers of no single dataset (STUDYID, USUBJID, VISITNUM, SPDEVID, NHOID) are not listed.',
    'Worked examples in the IG are not quoted, only Assumptions, §4.1.9 and CDISC Notes.',
    '',
    'SHEETS',
    'Statements — one row per quoted statement.',
    'Variables — one row per variable in Variables_Named: the SDTMIG v3.4 variable-table row as published,',
    '  plus Named_In. An empty CDISC CT Codelist Code(s) means the table gives no codelist (e.g. PCTESTCD).',
    'Provenance — versions; per sheet rows, columns and a content hash.',
    '',
    'COLUMNS — Statements',
    'Statement_ID: domain + A + assumption item (+ sub-item), e.g. IS-A7b; 4.1.9-n for the §4.1.9 passages in',
    '  order; <variable>-Notes for CDISC Notes. RS-DR = RS Disease Response Use Case Assumptions.',
    'Selected_As: facet | routing — why the row was selected.',
    'Source: the document and its date.',
    'Section: the numbered IG section; for Assumptions also the Assumptions heading.',
    'Item: assumption item and lettered sub-item. An item row holds the item\'s text before its first sub-item;',
    '  a sub-item row keeps its roman sub-items (e.g. 1c holds i-iv).',
    'Page: page in the IG (printed page = PDF page). Empty for CDISC Notes (CSV).',
    'Domain: the dataset whose Assumptions or Notes the text belongs to; empty for §4.1.9.',
    'Variables_Named: words that are variables of the dataset their first two letters name, in the SDTMIG v3.4',
    '  variable table; in order of first appearance.',
    'Model_Variables_Named: words not in that dataset\'s SDTMIG v3.4 table whose -- form is an SDTM v2.0 variable',
    '  of the dataset\'s observation class (e.g. LBPOS and LBLOC in LB-A8: --POS and --LOC, Findings).',
    'Domains_Named: dataset codes written as (XX) or "XX domain"; in order of first appearance.',
    'Text: the statement, verbatim.',
    '',
    'COLUMNS — Variables',
    'The columns of SDTMIG_v3.4.csv, unchanged, plus Named_In: the Statement_IDs that name the variable.',
    '',
    'READING',
    'Read with keep_default_na=False (pandas) or the equivalent, so that no cell is read as missing.',
]

wb = Workbook()
ws = wb.active
ws.title = 'ReadMe'
for i, text in enumerate(readme_lines, 1):
    c = ws.cell(row=i, column=1, value=text)
    c.font = HEADER_FONT if i == 1 else Font(name='Arial', size=10, bold=text.isupper() and bool(text))
    if i == 1:
        c.fill = BAND
ws.column_dimensions['A'].width = 80


def write_sheet(name, df):
    ws = wb.create_sheet(name)
    for ci, col in enumerate(df.columns, 1):
        c = ws.cell(row=1, column=ci, value=col)
        c.font, c.fill, c.border, c.alignment = HEADER_FONT, GREY, BORDER, WRAP
    for ri, row in enumerate(df.itertuples(index=False, name=None), 2):
        for ci, v in enumerate(row, 1):
            c = ws.cell(row=ri, column=ci, value=v)
            c.font, c.fill, c.border, c.alignment = DATA_FONT, PALE, BORDER, WRAP
    for ci, col in enumerate(df.columns, 1):
        width = max([len(col)] + [len(str(v)) for v in df[col]]) + 2
        ws.column_dimensions[ws.cell(row=1, column=ci).column_letter].width = min(width, WIDE.get(col, 50))
    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = ws.dimensions


write_sheet('Statements', statements)
write_sheet('Variables', variables)
OUTPUT_FILE.parent.mkdir(exist_ok=True)
wb.save(OUTPUT_FILE)
print(f'Written: {OUTPUT_FILE.relative_to(REPO_ROOT)}')

Written: sdtm-domain-reference/machine_actionable/SDTMIG_Statements.xlsx


In [7]:
# ── Provenance sheet ──
# Fixed keys for programs: the pinned source versions, and per data sheet its row count, column
# names and a content hash. The hash is computed from the file as saved, read back as text, so a
# consumer can recompute it; the xlsx file hash changes on every save, the content hash only when
# the data change.
PROVENANCE_VERSION_KEYS = ['SDTM_CT', 'COSMoS', 'SDTMIG', 'SDTM', 'NCIt']
PROVENANCE_METHOD = ('SHA-256 (UTF-8) of the sheet read as text, empty cells as "" (pandas read_excel, dtype=str): '
                     'header and rows, cells joined by tab, rows by newline. Equal hash = equal data.')


def sheet_sha256(df):
    lines = ['\t'.join(df.columns)] + ['\t'.join(row) for row in df.itertuples(index=False, name=None)]
    return hashlib.sha256('\n'.join(lines).encode('utf-8')).hexdigest()


def add_provenance(path, notebook):
    """Add a Provenance sheet (Key, Value) to a saved workbook."""
    sheets = pd.read_excel(path, sheet_name=None, dtype=str, keep_default_na=False)
    if 'Provenance' in sheets:
        raise RuntimeError(f'{path.name} already has a Provenance sheet')
    rows = [(k, VERSIONS[k]) for k in PROVENANCE_VERSION_KEYS]
    rows += [('Generated', f'{datetime.now():%Y-%m-%d %H:%M}'), ('Notebook', notebook)]
    for name, df in sheets.items():
        if name == 'ReadMe':
            continue
        rows += [(f'{name}.Rows', str(len(df))), (f'{name}.Columns', '|'.join(df.columns)),
                 (f'{name}.Content_SHA256', sheet_sha256(df))]
    rows.append(('Content_SHA256.Method', PROVENANCE_METHOD))
    wb_p = load_workbook(path)
    ws_p = wb_p.create_sheet('Provenance')
    for ci, head in enumerate(['Key', 'Value'], 1):
        cell = ws_p.cell(row=1, column=ci, value=head)
        cell.font = HEADER_FONT
        cell.fill = GREY
    for ri, (k, v) in enumerate(rows, 2):
        ws_p.cell(row=ri, column=1, value=k).font = DATA_FONT
        ws_p.cell(row=ri, column=2, value=v).font = DATA_FONT
    ws_p.column_dimensions['A'].width = 34
    ws_p.column_dimensions['B'].width = 90
    ws_p.freeze_panes = 'A2'
    wb_p.save(path)
    print(f'Provenance: {path.name}, {len(rows)} keys')


add_provenance(OUTPUT_FILE, 'sdtm-domain-reference/notebooks/SDTMIG_Statements.ipynb')

Provenance: SDTMIG_Statements.xlsx, 14 keys


In [8]:
# Read back: the written sheets equal the frames.
back = pd.read_excel(OUTPUT_FILE, sheet_name=None, dtype=str, keep_default_na=False)
for name, df in [('Statements', statements), ('Variables', variables)]:
    if not back[name].equals(df.astype(str)):
        raise RuntimeError(f'{name}: read-back differs from what was written')
print('Read-back: Statements and Variables as written.')

Read-back: Statements and Variables as written.
